<a href="https://colab.research.google.com/github/FrancoPanico/point-gesture-recognition-model/blob/main/Model_V1_PW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!ls -la /content/drive/MyDrive/pw2_processed/run_qf8aq2bi

Mounted at /content/drive
total 235
drwx------ 5 root root   4096 Nov 25 22:57 .
drwx------ 3 root root   4096 Nov 25 22:57 ..
-rw------- 1 root root   6010 Nov 25 22:58 data_health.csv
-rw------- 1 root root    317 Nov 25 22:58 diagnostics_detail.csv
-rw------- 1 root root    392 Nov 25 22:58 diagnostics_summary.csv
-rw------- 1 root root   8817 Nov 25 22:58 dropped_gestures.csv
-rw------- 1 root root  66267 Nov 25 22:58 gesture_duration_histogram.png
-rw------- 1 root root    310 Nov 25 22:58 gesture_duration_statistics.csv
-rw------- 1 root root 122027 Nov 25 22:58 gestures.csv
-rw------- 1 root root   1341 Nov 25 22:58 participant_gesture_statistics.csv
-rw------- 1 root root    337 Nov 25 22:58 participant_split.csv
-rw------- 1 root root  11318 Nov 25 22:58 quality_report.csv
drwx------ 2 root root   4096 Nov 25 22:59 resnet
drwx------ 2 root root   4096 Nov 25 22:57 windows
drwx------ 2 root root   4096 Nov 25 22:58 wrist_plots


In [2]:
%%writefile train.py
import argparse
from pathlib import Path
import numpy as np
from scipy.ndimage import median_filter
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def load_split_recordings(data_path: Path):
    windows_dir = data_path / "windows" if (data_path / "windows").is_dir() else data_path
    npz_files = sorted(windows_dir.glob("*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"Nessun file .npz trovato in {windows_dir}")

    splits = {"train": [], "val": [], "test": []}
    split_map = {"train": "train", "validation": "val", "val": "val", "test": "test"}

    for npz_file in npz_files:
        d = np.load(npz_file, allow_pickle=True)
        raw_split = str(d["split"].item()).strip().lower()
        split_key = split_map[raw_split]
        rec_name = str(d["recording_id"].item()) if "recording_id" in d else npz_file.stem

        splits[split_key].append({
            "name": rec_name,
            "X": d["X"].astype(np.float32),
            "y": d["y"].astype(np.float32),
            "overlaps": d["overlaps_gesture"].astype(bool)
            if "overlaps_gesture" in d
            else np.zeros(len(d["y"]), dtype=bool),
        })

    for k, v in splits.items():
        print(f"Split '{k}': {len(v)} registrazioni, {sum(len(r['y']) for r in v)} finestre totali")
    return splits


def extract_features(X: np.ndarray) -> np.ndarray:
    N, T, J, C = X.shape
    pos = X.reshape(N, T, J * C)
    vel = np.diff(pos, axis=1, prepend=pos[:, :1, :])
    return np.concatenate([pos, vel], axis=-1)


def prepare_datasets(splits):
    X_train_list, y_train_list = [], []
    for rec in splits["train"]:
        keep = (rec["y"] == 1) | (~rec["overlaps"])
        if keep.any():
            X_train_list.append(extract_features(rec["X"][keep]))
            y_train_list.append(rec["y"][keep])

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)

    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
    X_train = (X_train - mean) / std

    eval_splits = {"val": [], "test": []}
    for split in ["val", "test"]:
        for rec in splits[split]:
            feat = (extract_features(rec["X"]) - mean) / std
            eval_splits[split].append({"name": rec["name"], "X": feat, "y": rec["y"]})

    return X_train, y_train, eval_splits, mean, std


class PointingCNNBiGRU(nn.Module):
    def __init__(self, in_features: int = 192, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_features, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.25),
        )
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        h = self.conv(x.transpose(1, 2)).transpose(1, 2)
        h, _ = self.gru(h)
        center_feat = h[:, 8:12, :].mean(dim=1)
        global_feat = h.mean(dim=1)
        return self.head(torch.cat([center_feat, global_feat], dim=-1)).squeeze(-1)


@torch.no_grad()
def predict_recordings(model, recordings, device, smooth_k: int = 3):
    model.eval()
    all_probs, all_targets, per_rec = [], [], []
    for rec in recordings:
        x_t = torch.from_numpy(rec["X"]).to(device)
        probs = torch.sigmoid(model(x_t)).cpu().numpy()
        if smooth_k > 1 and len(probs) >= smooth_k:
            probs = median_filter(probs, size=smooth_k, mode="nearest")
        all_probs.append(probs)
        all_targets.append(rec["y"])
        per_rec.append((rec["name"], probs, rec["y"]))
    return np.concatenate(all_probs), np.concatenate(all_targets), per_rec


def find_best_threshold(probs, targets):
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.arange(0.30, 0.81, 0.02):
        preds = (probs >= tau).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            targets, preds, average="binary", zero_division=0
        )
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
    return best_tau, best_f1


def compute_event_metrics(per_rec, tau: float):
    tp_events, total_gt_events, fp_events = 0, 0, 0
    for _, probs, y in per_rec:
        pred = (probs >= tau).astype(int)
        gt = (y == 1).astype(int)

        def get_segments(arr):
            padded = np.pad(arr, (1, 1))
            return list(zip(np.where(np.diff(padded) == 1)[0], np.where(np.diff(padded) == -1)[0]))

        gt_segs = get_segments(gt)
        pred_segs = get_segments(pred)
        total_gt_events += len(gt_segs)
        for gs, ge in gt_segs:
            if pred[gs:ge].sum() > 0:
                tp_events += 1
        for ps, pe in pred_segs:
            if gt[ps:pe].sum() == 0:
                fp_events += 1

    recall = tp_events / max(total_gt_events, 1)
    precision = tp_events / max(tp_events + fp_events, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    return precision, recall, f1, tp_events, total_gt_events, fp_events


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_path", type=str, required=True)
    parser.add_argument("--save_path", type=str, default="best_model.pt")
    parser.add_argument("--epochs", type=int, default=60)
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--patience", type=int, default=12)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    splits = load_split_recordings(Path(args.data_path))
    X_train, y_train, eval_splits, mean, std = prepare_datasets(splits)
    print(f"Train windows (filtered): {X_train.shape} | Positives: {int(y_train.sum())}/{len(y_train)}")

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=args.batch_size,
        shuffle=True,
    )

    pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], device=device)
    model = PointingCNNBiGRU(in_features=X_train.shape[-1]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    best_val_f1, best_tau, epochs_no_improve = -1.0, 0.5, 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(by)

        val_probs, val_targets, _ = predict_recordings(model, eval_splits["val"], device, smooth_k=3)
        tau, val_f1 = find_best_threshold(val_probs, val_targets)
        val_prauc = average_precision_score(val_targets, val_probs)
        scheduler.step(val_f1)

        print(
            f"Epoch {epoch:02d} | Loss: {total_loss / len(y_train):.4f} | "
            f"Val F1: {val_f1:.4f} (tau={tau:.2f}) | Val PR-AUC: {val_prauc:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1, best_tau, epochs_no_improve = val_f1, tau, 0
            torch.save(
                {"model_state": model.state_dict(), "tau": best_tau, "mean": mean, "std": std},
                args.save_path,
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= args.patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    ckpt = torch.load(args.save_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    best_tau = ckpt["tau"]

    test_probs, test_targets, test_per_rec = predict_recordings(
        model, eval_splits["test"], device, smooth_k=3
    )
    test_preds = (test_probs >= best_tau).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        test_targets, test_preds, average="binary", zero_division=0
    )
    prauc = average_precision_score(test_targets, test_probs)
    ep, er, ef1, tp_e, tot_e, fp_e = compute_event_metrics(test_per_rec, best_tau)

    print("\n=== FINAL TEST RESULTS (3 Unseen Subjects) ===")
    print(f"Optimal Threshold (from Val): {best_tau:.2f}")
    print(f"Window-Level -> Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f} | PR-AUC: {prauc:.4f}")
    print(
        f"Event-Level  -> Precision: {ep:.4f} | Recall: {er:.4f} | F1: {ef1:.4f} "
        f"(Detected {tp_e}/{tot_e} gestures, {fp_e} false alarms)"
    )


if __name__ == "__main__":
    main()

Writing train.py


In [3]:
!python train.py \
  --data_path /content/drive/MyDrive/pw2_processed/run_qf8aq2bi \
  --save_path /content/drive/MyDrive/pw2_processed/run_qf8aq2bi/best_model.pt

Using device: cuda
Split 'train': 47 registrazioni, 18206 finestre totali
Split 'val': 9 registrazioni, 3510 finestre totali
Split 'test': 9 registrazioni, 3786 finestre totali
Train windows (filtered): (12509, 20, 192) | Positives: 10424/12509
Epoch 01 | Loss: 0.1124 | Val F1: 0.7767 (tau=0.50) | Val PR-AUC: 0.7965
Epoch 02 | Loss: 0.0505 | Val F1: 0.7780 (tau=0.80) | Val PR-AUC: 0.8124
Epoch 03 | Loss: 0.0344 | Val F1: 0.7864 (tau=0.72) | Val PR-AUC: 0.7803
Epoch 04 | Loss: 0.0269 | Val F1: 0.7730 (tau=0.68) | Val PR-AUC: 0.7482
Epoch 05 | Loss: 0.0188 | Val F1: 0.7665 (tau=0.36) | Val PR-AUC: 0.7571
Epoch 06 | Loss: 0.0191 | Val F1: 0.7700 (tau=0.56) | Val PR-AUC: 0.7688
Epoch 07 | Loss: 0.0133 | Val F1: 0.7888 (tau=0.78) | Val PR-AUC: 0.8047
Epoch 08 | Loss: 0.0146 | Val F1: 0.7753 (tau=0.44) | Val PR-AUC: 0.7819
Epoch 09 | Loss: 0.0128 | Val F1: 0.7691 (tau=0.38) | Val PR-AUC: 0.7777
Epoch 10 | Loss: 0.0121 | Val F1: 0.7692 (tau=0.52) | Val PR-AUC: 0.7574
Epoch 11 | Loss: 0.0059 |